In [1]:
import numpy as np
import torch
import pprint
import onnx
import copy
import hls4ml
from torch.utils.data import DataLoader
from utils.utils import read_object
from hls4ml.model.profiling import numerical
from onnx import numpy_helper
from cnn_base import t1_quantizedCNN
from sklearn.metrics import accuracy_score
%matplotlib inline
torch.manual_seed(4)

def print_dict(d, indent=0):
    for key, value in d.items():
        print('  ' * indent + str(key), end='')
        if isinstance(value, dict):
            print()
            print_dict(value, indent + 1)
        else:
            print(':' + ' ' * (20 - len(key) - 2 * indent) + str(value))

def trace_layers(config):
    config['Model']['TraceOutput'] = True
    for layer in config['LayerName']:
        config['LayerName'][layer]['Trace'] = True

def ensure_layer(name):
    layer_cfg.setdefault(name, {})
    return layer_cfg[name]

def ensure_precision(name):
    layer = ensure_layer(name)
    layer.setdefault('Precision', {})
    return layer['Precision']


DEBUG = False

In [2]:
onnx_model_path = "./hls4ml_onnxs/sat6-cnn-t1w8_teste.onnx"
batch_size = 512
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_dataset_output_name = "./dataset/test_dataset.pkl"

dataset_test = read_object(test_dataset_output_name)
loader_test = DataLoader(dataset_test, batch_size, shuffle=False)

X_test = dataset_test.tensors[0].numpy().astype(np.float32)
y_test = dataset_test.tensors[1].numpy().astype(np.int32)


if DEBUG:
    onnx_model = onnx.load(onnx_model_path)

    print("=== Nós do grafo (nome, tipo) ===")
    for node in onnx_model.graph.node:
        print(f"{node.name:25s} -> {node.op_type}")

    print("\n=== Input ===")
    for inp in onnx_model.graph.input:
        print(inp.name, [d.dim_value for d in inp.type.tensor_type.shape.dim])

In [3]:
onnx_model = onnx.load(onnx_model_path)

config = hls4ml.utils.config_from_onnx_model(
    onnx_model, granularity='name', backend='Vitis',
    default_precision='ap_fixed<16,6>'
)

hls_model = hls4ml.converters.convert_from_onnx_model(
    onnx_model,
    output_dir='./hls4ml_prjs/hls4ml_prj_t1_v4',
    io_type='io_stream', backend='Vitis',
    hls_config=config, part='xc7z020clg400-1',
)

FileNotFoundError: [Errno 2] No such file or directory: './hls4ml_onnxs/sat6-cnn-t1w8_teste.onnx'

In [ ]:
real_config = hls_model.config.config
layer_cfg = real_config['HLSConfig']['LayerName']


for layer in ['Conv2D_Conv_0', 'Conv2D_Conv_1', 'Dense_MatMul_0']:
    p = ensure_precision(layer)
    p['weight'] = 'ap_int<8>'
    ensure_layer(layer)['Strategy'] = 'Resource'

ensure_precision('Conv2D_Conv_0')['accum']  = 'ap_fixed<24,24>'
ensure_precision('Conv2D_Conv_1')['accum']  = 'ap_fixed<24,24>'
ensure_precision('Dense_MatMul_0')['accum'] = 'ap_fixed<40,26>'

for layer, prec in [
    ('Quant_0_rescale', 'ap_fixed<16,9>'),
    ('Quant_1_rescale', 'ap_fixed<16,8>'),
    ('Quant_2_rescale', 'ap_fixed<16,9>'),
    ('Quant_3_scale',   'ap_ufixed<16,8>'),
    ('Quant_4_scale',   'ap_ufixed<16,8>'),
    ('Quant_4_rescale', 'ap_ufixed<16,8>'),
]:
    ensure_layer(layer)['Precision'] = prec

ensure_precision('Relu_0')['result'] = 'ap_ufixed<22,8>'
ensure_precision('Relu_1')['result'] = 'ap_ufixed<22,8>'
ensure_precision('global_in')['result'] = 'ap_uint<8>'

if 'Quant_3_act' in layer_cfg:
    ensure_precision('Quant_3_act')['result'] = 'ap_int<8>'
if 'Quant_4_act' in layer_cfg:
    ensure_precision('Quant_4_act')['result'] = 'ap_int<8>'

trace_layers(real_config.get('HLSConfig'))
real_config

In [ ]:
config.get('Model')

In [ ]:
hls_model.config.config

In [ ]:
if DEBUG:
    hls_model.compile()
    numerical(model=onnx_model, hls_model=hls_model, X=X_test[:2000], plot='boxplot')

In [ ]:
hls_model.compile()
model = t1_quantizedCNN(8)
model.load_state_dict(torch.load('pytorch_models/sat6-cnn-t1.pt', map_location=device))
model.eval()
with torch.no_grad():
    y_ref = model(torch.FloatTensor(X_test)).numpy()

y_hls = hls_model.predict(np.ascontiguousarray(X_test))

print('Accuracy (Brevitas float): {}'.format(accuracy_score(y_test, np.argmax(y_ref, axis=1))))
print('Accuracy (HLS): {}'.format(accuracy_score(y_test, np.argmax(y_hls, axis=1))))

In [ ]:
hls_model.build(csim=False, synth=True)